In [3]:
import numpy as np
import time

from sklearn.datasets import fetch_openml 
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

In [5]:
fashion_mnist = fetch_openml('Fashion-MNIST', version=1,as_frame=False)
X = fashion_mnist.data
y = fashion_mnist.target.astype(int)

print(f"Total Dataset Size: {X.shape[0]} images, each with {X.shape[1]} pixels.")

Total Dataset Size: 70000 images, each with 784 pixels.


In [7]:
X_subset, _, y_subset, _ = train_test_split(
    X,y,
    train_size=12000,
    stratify=y,
    random_state=42
)

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X_subset, y_subset,
    test_size=2000,
    stratify=y_subset,
    random_state=42
)
print(f"Training Images: {X_train.shape[0]}")
print(f"Testing Images: {X_test.shape[0]}")

Training Images: 10000
Testing Images: 2000


In [12]:
print(f"Before Scaling -> Min: {X_train.min()}, Max: {X_train.max()}")
X_train = X_train / 255.0
X_test = X_test / 255.0
print(f"After Scaling -> Min: {X_train.min()}, Max: {X_train.max()}")

Before Scaling -> Min: 0.0, Max: 1.0
After Scaling -> Min: 0.0, Max: 0.00392156862745098


In [16]:
k_values = [1,3,5,7,9,15]
results= {}
print(f"{'K Value':<8} | {'Accuracy':<10} | {'Prediction Time (Seconds)':<25}")
print("-" * 50)
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, metric='euclidean',n_jobs=-1)
    knn.fit(X_train,y_train)
    start_time = time.time()
    y_pred = knn.predict(X_test)
    elapsed_time = time.time() - start_time

    acc = accuracy_score(y_test,y_pred)
    results[k] = {
        "Accuracy": acc,
        "Time": elapsed_time,
        "Predictions": y_pred
    }
    print(f"{k:<8} | {acc * 100:<9.2f}% | {elapsed_time:<25.2f}")

K Value  | Accuracy   | Prediction Time (Seconds)
--------------------------------------------------
1        | 79.30    % | 0.21                     
3        | 80.90    % | 0.20                     
5        | 81.00    % | 0.20                     
7        | 81.10    % | 0.21                     
9        | 80.95    % | 0.21                     
15       | 80.25    % | 0.21                     


In [18]:
class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle Boot"
]
best_k = max(results, key=lambda k: results[k]["Accuracy"])
print(f"Best K is: {best_k} with {results[best_k]['Accuracy']* 100:.2f}% Accuracy\n")
print("Per-Class Classification Report:")
print(classification_report(y_test, results[best_k]["Predictions"], target_names=class_names))

Best K is: 7 with 81.10% Accuracy

Per-Class Classification Report:
              precision    recall  f1-score   support

 T-shirt/top       0.73      0.83      0.78       200
     Trouser       0.97      0.94      0.96       200
    Pullover       0.67      0.73      0.70       200
       Dress       0.85      0.83      0.84       200
        Coat       0.74      0.69      0.72       200
      Sandal       1.00      0.76      0.86       200
       Shirt       0.58      0.55      0.56       200
     Sneaker       0.81      0.92      0.86       200
         Bag       0.97      0.94      0.95       200
  Ankle Boot       0.85      0.94      0.89       200

    accuracy                           0.81      2000
   macro avg       0.82      0.81      0.81      2000
weighted avg       0.82      0.81      0.81      2000

